# Probability/nonprobability

**STA 503 1.0 Sampling Techniques** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta503.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Probability versus nonprobability selection](#theory-01)
- [Simple random sampling](#theory-02)
- [Lottery, random-number tables and software](#theory-03)
- [Remainder methods](#theory-04)
- [Stratified, systematic and cluster designs](#theory-05)
- [Inclusion and weights](#theory-06)
- [Design choice](#theory-07)

## Detailed theory · Probability sampling and selection procedures

<a id="theory-01"></a>

### Probability versus nonprobability selection
In a probability design each in-scope unit has a known positive inclusion probability, and the random mechanism defines possible samples. Nonprobability approaches such as convenience, judgement, quota and snowball sampling may be useful for access or exploration but do not supply ordinary design-based inclusion probabilities. A large convenience sample is not an SRS.

<a id="theory-02"></a>

### Simple random sampling
Without replacement, every unordered sample of n distinct units from N is equally likely, with sample probability $1/\binom Nn$ and unit inclusion probability n/N. With replacement, draws can repeat and selection probabilities concern draws as well as eventual unit inclusion. These are different designs.

<a id="theory-03"></a>

### Lottery, random-number tables and software
Give frame units unique labels and use a genuinely random mechanism. For a digit table, state the starting point, reading direction, block size, valid label range and whether repeated labels are rejected. Ignore out-of-range blocks and repeats under SRS without replacement. In software select n distinct labels without replacement and record the seed and frame version for reproducibility.

<a id="theory-04"></a>

### Remainder methods
Mapping equally likely digit blocks to remainders is unbiased only if every target label has equal numbers of accepted preimages. If the digit-block range is not a multiple of N, reject the excess top range before taking remainders. Plain modulo reduction can overrepresent some labels. Treat a remainder-zero label consistently with the numbering convention.

<a id="theory-05"></a>

### Stratified, systematic and cluster designs
Stratification partitions units and selects within every stratum. Systematic sampling uses a random start and a fixed interval; hidden periodicity can distort representation and variance estimation. Cluster sampling selects groups and observes all or some elements within them. Multistage designs combine levels; the final data can require weights and cluster-aware uncertainty.

<a id="theory-06"></a>

### Inclusion and weights
For unit inclusion probability $\pi_i$, a basic design weight is $w_i=1/\pi_i$. Under suitable conditions a total estimator is $\hat\tau=\sum_{i\in s}y_i/\pi_i$. Equal weights are not justified merely because every record occupies one row. Nonresponse and calibration adjustments need their own stated assumptions.

<a id="theory-07"></a>

### Design choice
Compare frame availability, geography, cost, subgroup precision, expected heterogeneity and logistics. Probability designs are not interchangeable even when marginal inclusion probabilities happen to agree. State the selection procedure and use the variance estimator matching it.

## Concepts covered

**Course points:** Probability/nonprobability; SRS; systematic; stratified; cluster; multistage and PPS sampling.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Sampling designs: show who can be selected

### Question and assumptions

Compare an SRS and a stratified sample from six labelled households, split into two strata of three.

### Procedure

1. Enumerate all SRS pairs.
2. Enumerate all pairs obtained by choosing one household from each stratum.
3. Compare possible samples, not only sample size.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Design | Possible samples | Count |
| --- | --- | --- |
| SRS of two | AB, AC, AD, AE, AF, BC, BD, BE, BF, CD, CE, CF, DE, DF, EF | 15 |
| One per stratum | AD, AE, AF, BD, BE, BF, CD, CE, CF | 9 |

| Design | Inclusion probability per household | Important distinction |
| --- | --- | --- |
| SRS | 0.333333 | All pairs equally possible |
| Stratified | 0.333333 | Within-stratum pairs impossible |
| Convenience | Unknown | No design-based inclusion probability |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


population=list("ABCDEF"); strata=[list("ABC"),list("DEF")]
srs=list(combinations(population,2)); stratified=list(product(*strata))
show_table(["Design","Possible samples","Count"],[["SRS of two",", ".join("".join(s) for s in srs),len(srs)],["One per stratum",", ".join("".join(s) for s in stratified),len(stratified)]])
show_table(["Design","Inclusion probability per household","Important distinction"],[["SRS",2/6,"All pairs equally possible"],["Stratified",1/3,"Within-stratum pairs impossible"],["Convenience","Unknown","No design-based inclusion probability"]])
assert len(srs)==15 and len(stratified)==9


### Decision and interpretation

Equal marginal inclusion probabilities do not imply identical sampling designs. The possible sample sets differ, so the variance calculation must match the design.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
sizes=np.array([700,500,300,100]); mus=np.array([62,68,74,82]); grp=np.concatenate([[i]*v for i,v in enumerate(sizes)])
y=np.concatenate([rng.normal(mus[i],10,v) for i,v in enumerate(sizes)]); N=len(y); n=80; W=sizes/N
srs=[y[rng.choice(N,n,False)].mean() for _ in range(700)]; strat=[]
for _ in range(700):
 a=[rng.choice(y[grp==i],max(1,round(n*W[i])),False).mean() for i in range(4)]; strat.append(np.sum(W*a))
print('SRS SE',np.std(srs).round(2),'stratified SE',np.std(strat).round(2))
u=np.array([1,2,3,4,10,20]); print('PPS chances',np.round(u/u.sum(),3))
plt.hist(srs,alpha=.5,label='SRS'); plt.hist(strat,alpha=.5,label='stratified'); plt.legend(); plt.show()

### Practice set · Sampling methods
1. A list of 2,000 households has a periodic order by neighborhood. Is systematic sampling automatically safe? Why?
2. Compare cluster and stratified sampling for a national school survey.
3. A sample selects districts, then schools, then students. Name the stages and likely design concern.
4. What makes a probability sample different from a large social-media poll?

<details><summary>Answer sketches</summary>
1. No; periodicity may align with the interval. Randomize list order or assess periodic structure. 2. Stratify to ensure representation and precision across groups; cluster to save travel, with possible design effect from within-school similarity. 3. Multistage sampling; account for unequal inclusion probabilities and clustered errors. 4. Known nonzero selection probabilities support design-based estimation; poll participation may be self-selected and unknown.
</details>

### Debrief
Choose a design for scattered units and explain cost vs precision.

Record the formula, assumptions, and one follow-up question.